# 07 — Leakage Ablation (Optional, Strongest Evidence Piece)

**Goal of this notebook**: directly demonstrate *why* the patient-level split matters,
by training the same DenseNet121 architecture two ways — a naive slice-level (random)
split, the same mistake the public Kaggle version of this dataset makes, vs. our
patient-level split — and comparing test accuracy side by side.

**Why this is worth doing**: without this notebook, the leakage fix is invisible —
just a best practice you did quietly. With it, you have a concrete, visible finding:
"naive split reports X% accuracy; once leakage is fixed, real accuracy is Y%." That's a
genuinely strong thing to show in a portfolio or report — it demonstrates you understand
*why* the methodology matters, not just that you followed a checklist.

In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

from src.data_utils import SplitConfig, build_tf_dataset, patient_level_split
from src.evaluate import evaluate_predictions
from src.models import build_transfer_model, unfreeze_for_finetuning
from src.train import train_model

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

## Build the naive (leaky) split for comparison

**Intent**: deliberately reproduce the Kaggle dataset's mistake — split at the image
level with `train_test_split`, ignoring `patient_id` entirely — so we have a fair
apples-to-apples comparison against our patient-level split.

In [ ]:
metadata = pd.read_csv("../data/processed/metadata.csv")

train_idx, temp_idx = train_test_split(
    metadata.index, test_size=0.3, stratify=metadata["label"], random_state=42
)
val_idx, test_idx = train_test_split(
    metadata.loc[temp_idx].index,
    test_size=0.5,
    stratify=metadata.loc[temp_idx, "label"],
    random_state=42,
)

naive_split = metadata.copy()
naive_split["split"] = "unassigned"
naive_split.loc[train_idx, "split"] = "train"
naive_split.loc[val_idx, "split"] = "val"
naive_split.loc[test_idx, "split"] = "test"

# Confirm this naive split DOES leak patients across sets — that's the point
overlap = naive_split.groupby("patient_id")["split"].nunique()
n_leaking = (overlap > 1).sum()
print(
    f"Patients spanning multiple splits in the naive split: {n_leaking} "
    f"(this is expected — it's the bug we're demonstrating)"
)

## Patient-level (clean) split, for comparison

In [ ]:
clean_split = patient_level_split(metadata, SplitConfig(test_size=0.3, val_size=0.15, seed=42))

## Train DenseNet121 on the naive split

In [ ]:
def to_3channel(images, labels):
    return tf.repeat(images, repeats=3, axis=-1), labels


def make_datasets(split_df):
    train_ds = build_tf_dataset(
        split_df, "train", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True
    ).map(to_3channel)
    val_ds = build_tf_dataset(
        split_df, "val", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
    ).map(to_3channel)
    test_ds = build_tf_dataset(
        split_df, "test", CLASS_NAMES, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False
    ).map(to_3channel)
    return train_ds, val_ds, test_ds


naive_train, naive_val, naive_test = make_datasets(naive_split)

naive_model = build_transfer_model(
    input_shape=(*IMG_SIZE, 3), num_classes=len(CLASS_NAMES), freeze_base=True
)
train_model(
    naive_model,
    naive_train,
    naive_val,
    run_name="ablation_naive_split_phase1",
    epochs=10,
    checkpoint_dir=Path("../models/saved_models/ablation"),
)
naive_model = unfreeze_for_finetuning(naive_model, num_layers_to_unfreeze=30)
train_model(
    naive_model,
    naive_train,
    naive_val,
    run_name="ablation_naive_split_phase2",
    epochs=10,
    checkpoint_dir=Path("../models/saved_models/ablation"),
)

## Train DenseNet121 on the clean (patient-level) split

**Intent**: identical architecture, identical hyperparameters, identical epoch budget —
the *only* difference from the cell above is which split strategy produced the data.
That isolation is what makes this a fair ablation.

In [ ]:
clean_train, clean_val, clean_test = make_datasets(clean_split)

clean_model = build_transfer_model(
    input_shape=(*IMG_SIZE, 3), num_classes=len(CLASS_NAMES), freeze_base=True
)
train_model(
    clean_model,
    clean_train,
    clean_val,
    run_name="ablation_clean_split_phase1",
    epochs=10,
    checkpoint_dir=Path("../models/saved_models/ablation"),
)
clean_model = unfreeze_for_finetuning(clean_model, num_layers_to_unfreeze=30)
train_model(
    clean_model,
    clean_train,
    clean_val,
    run_name="ablation_clean_split_phase2",
    epochs=10,
    checkpoint_dir=Path("../models/saved_models/ablation"),
)

## Compare test accuracy: naive vs. patient-level split

**Intent**: this is the headline result of the whole ablation — the accuracy gap here
*is* the leakage effect, made visible and measurable rather than just asserted.

In [ ]:
def get_predictions(model, dataset):
    y_true_idx, y_pred_idx = [], []
    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_pred_idx.extend(preds.argmax(axis=1))
        y_true_idx.extend(labels.numpy())
    y_true = np.array([CLASS_NAMES[i] for i in y_true_idx])
    y_pred = np.array([CLASS_NAMES[i] for i in y_pred_idx])
    return y_true, y_pred


y_true_naive, y_pred_naive = get_predictions(naive_model, naive_test)
y_true_clean, y_pred_clean = get_predictions(clean_model, clean_test)

results_naive = evaluate_predictions(y_true_naive, y_pred_naive, CLASS_NAMES)
results_clean = evaluate_predictions(y_true_clean, y_pred_clean, CLASS_NAMES)

ablation_summary = pd.DataFrame(
    {
        "Naive (leaky) split": [results_naive["report"].loc["accuracy"].iloc[0]],
        "Patient-level (clean) split": [results_clean["report"].loc["accuracy"].iloc[0]],
    },
    index=["Test accuracy"],
)

ablation_summary

## Conclusion

Fill in after running: report the accuracy gap in plain terms, e.g. *"The naive
slice-level split reported X% test accuracy; the same architecture, trained identically
but evaluated on a truly held-out set of patients, achieved Y%. The Z-point gap is the
leakage effect — accuracy the naive split reported that doesn't reflect real
generalization to new patients."*

This conclusion (with your actual numbers) belongs in `README.md`'s Results section and
is worth surfacing explicitly on the dashboard's Model Performance page.